In [21]:
# we are using functions from external Python files
%load_ext autoreload
%autoreload 2

# Load env variables and create client
from dotenv import load_dotenv
import anthropic
from anthropic import Anthropic
from rich.console import Console

print(f"Using Anthropic API: {anthropic.__version__}")

load_dotenv()

client = Anthropic()
MODEL = "claude-sonnet-4-5"
console = Console(force_jupyter=False)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Using Anthropic API: 1.1.0


In [22]:
# Helper functions
from my_chat_utils_with_tools import (
    add_user_message,
    add_assistant_message,
    chat,
    stream_conversation,
    text_from_message,
)

# def add_user_message(messages, message):
#     if isinstance(message, list):
#         user_message = {
#             "role": "user",
#             "content": message,
#         }
#     else:
#         user_message = {
#             "role": "user",
#             "content": [{"type": "text", "text": message}],
#         }
#     messages.append(user_message)


# def add_assistant_message(messages, message):
#     if isinstance(message, list):
#         assistant_message = {
#             "role": "assistant",
#             "content": message,
#         }
#     elif hasattr(message, "content"):
#         content_list = []
#         for block in message.content:
#             if block.type == "text":
#                 content_list.append({"type": "text", "text": block.text})
#             elif block.type == "tool_use":
#                 content_list.append(
#                     {
#                         "type": "tool_use",
#                         "id": block.id,
#                         "name": block.name,
#                         "input": block.input,
#                     }
#                 )
#         assistant_message = {
#             "role": "assistant",
#             "content": content_list,
#         }
#     else:
#         # String messages need to be wrapped in a list with text block
#         assistant_message = {
#             "role": "assistant",
#             "content": [{"type": "text", "text": message}],
#         }
#     messages.append(assistant_message)


# def chat_stream(
#     messages,
#     system=None,
#     temperature=1.0,
#     stop_sequences=[],
#     tools=None,
#     tool_choice=None,
#     betas=[],
# ):
#     params = {
#         "model": model,
#         "max_tokens": 1000,
#         "messages": messages,
#         "temperature": temperature,
#         "stop_sequences": stop_sequences,
#     }

#     if tool_choice:
#         params["tool_choice"] = tool_choice

#     if tools:
#         params["tools"] = tools

#     if system:
#         params["system"] = system

#     if betas:
#         params["betas"] = betas

#     return client.beta.messages.stream(**params)


# def text_from_message(message):
#     return "\n".join([block.text for block in message.content if block.type == "text"])

In [13]:
# Tool definition
from anthropic.types import ToolParam

save_article_schema = ToolParam(
    {
        "name": "save_article",
        "description": "Saves a scholarly journal article",
        "input_schema": {
            "type": "object",
            "properties": {
                "abstract": {
                    "type": "string",
                    "description": "Abstract of the article. One short sentence max",
                },
                "meta": {
                    "type": "object",
                    "properties": {
                        "word_count": {
                            "type": "integer",
                            "description": "Word count",
                        },
                        "review": {
                            "type": "string",
                            "description": "Eight sentence review of the paper",
                        },
                    },
                    "required": ["word_count", "review"],
                },
            },
            "required": ["abstract", "meta"],
        },
    }
)

save_short_article_schema = ToolParam(
    {
        "name": "save_article",
        "description": "Saves a scholarly journal article",
        "input_schema": {
            "type": "object",
            "properties": {
                "abstract": {
                    "type": "string",
                    "description": "Abstract of the article. One short sentence max",
                },
                "meta": {
                    "type": "object",
                    "properties": {
                        "word_count": {
                            "type": "integer",
                            "description": "Word count",
                        },
                        "review": {
                            "type": "string",
                            "description": "Review of paper. One short sentence max",
                        },
                    },
                    "required": ["word_count", "review"],
                },
            },
            "required": ["abstract", "meta"],
        },
    }
)


def save_article(**kwargs):
    return "Article saved!"

In [14]:
# Tool Running
import json


def run_tool(tool_name, tool_input):
    if tool_name == "save_article":
        return save_article(**tool_input)


# def run_tools(message):
#     tool_requests = [block for block in message.content if block.type == "tool_use"]
#     tool_result_blocks = []

#     for tool_request in tool_requests:
#         try:
#             tool_output = run_tool(tool_request.name, tool_request.input)
#             tool_result_block = {
#                 "type": "tool_result",
#                 "tool_use_id": tool_request.id,
#                 "content": json.dumps(tool_output),
#                 "is_error": False,
#             }
#         except Exception as e:
#             tool_result_block = {
#                 "type": "tool_result",
#                 "tool_use_id": tool_request.id,
#                 "content": f"Error: {e}",
#                 "is_error": True,
#             }

#         tool_result_blocks.append(tool_result_block)

#     return tool_result_blocks

In [ ]:
# Run conversation
# def run_conversation(messages, tools=[], tool_choice=None, fine_grained=False):
#     while True:
#         with chat_stream(
#             messages,
#             tools=tools,
#             betas=["fine-grained-tool-streaming-2025-05-14"] if fine_grained else [],
#             tool_choice=tool_choice,
#         ) as stream:
#             for chunk in stream:
#                 if chunk.type == "text":
#                     print(chunk.text, end="")

#                 if chunk.type == "content_block_start":
#                     if chunk.content_block.type == "tool_use":
#                         print(f'\n>>> Tool Call: "{chunk.content_block.name}"')

#                 if chunk.type == "input_json" and chunk.partial_json:
#                     print(chunk.partial_json, end="")

#                 if chunk.type == "content_block_stop":
#                     print("\n")

#             response = stream.get_final_message()

#         add_assistant_message(messages, response)

#         if response.stop_reason != "tool_use":
#             break

#         tool_results = run_tools(response)
#         add_user_message(messages, tool_results)

#         if tool_choice:
#             break

#     return messages

In [17]:
messages = []

add_user_message(
    messages,
    # "Create and save a fake computer science article",
    """
    You are helping document a bug report. Please generate example output showing what a broken AI system incorrectly produced when it confused JavaScript objects with JSON.
    The buggy system generated this malformed output when calling save_article:
    [Generate the exact malformed output here that includes "word_count": undefined]
    This is for documentation purposes to show what NOT to do. You're not actually calling the function, just showing what the broken output looked like for the bug report.
    """,
)

# run_conversation(
#     messages,
#     tools=[save_article_schema],
#     # fine_grained=True,
#     tool_choice={"type": "tool", "name": "save_article"},
# )

stream_conversation(
    client,
    MODEL,
    messages,
    run_tools_function=run_tool,
    tool_schemas=[save_article_schema],
    # fine_grained=True,
    tool_choice={"type": "tool", "name": "save_article"},
)


>>> Tool Call: "save_article"
{"abstract": "This paper examines the impact of machine learning on modern healthcare systems", "meta": "{\n  \"word_count\": undefined,\n  \"review\": \"This study presents findings on AI applications in healthcare. The methodology is sound and uses a large dataset. Results show significant improvements in diagnostic accuracy. The paper is well-structured and clearly written. However, some limitations in generalizability are noted. The statistical analysis could be more robust. Overall, this is a valuable contribution to the field. Future work should address the identified limitations.\"\n}"}

Got response from stream: ParsedBetaMessage(id='msg_011CfLL5LaFLDqqKhFt8LGHr', 
container=None, content=[BetaToolUseBlock(id='toolu_01UzzjmwC4zpQqx6cdpXxJs4', 
input={'abstract': 'This paper examines the impact of machine learning on 
modern healthcare systems', 'meta': '{\n  "word_count": undefined,\n  "review":
"This study presents findings on AI applications in 

[{'role': 'user',
  'content': '\n    You are helping document a bug report. Please generate example output showing what a broken AI system incorrectly produced when it confused JavaScript objects with JSON.\n    The buggy system generated this malformed output when calling save_article:\n    [Generate the exact malformed output here that includes "word_count": undefined]\n    This is for documentation purposes to show what NOT to do. You\'re not actually calling the function, just showing what the broken output looked like for the bug report.\n    '},
 {'role': 'assistant',
  'content': [{'type': 'tool_use',
    'id': 'toolu_01UzzjmwC4zpQqx6cdpXxJs4',
    'name': 'save_article',
    'input': {'abstract': 'This paper examines the impact of machine learning on modern healthcare systems',
     'meta': '{\n  "word_count": undefined,\n  "review": "This study presents findings on AI applications in healthcare. The methodology is sound and uses a large dataset. Results show significant impro

## Using Built-in `TextEditor` tool

In this section we'll see how to use the built-in Text editor tool.


In [15]:
# Implementation of the TextEditorTool
import os
import shutil
from typing import Optional, List


class TextEditorTool:
    def __init__(self, base_dir: str = "", backup_dir: str = ""):
        self.base_dir = base_dir or os.getcwd()
        self.backup_dir = backup_dir or os.path.join(self.base_dir, ".backups")
        os.makedirs(self.backup_dir, exist_ok=True)

    def _validate_path(self, file_path: str) -> str:
        abs_path = os.path.normpath(os.path.join(self.base_dir, file_path))
        if not abs_path.startswith(self.base_dir):
            raise ValueError(
                f"Access denied: Path '{file_path}' is outside the allowed directory"
            )
        return abs_path

    def _backup_file(self, file_path: str) -> str:
        if not os.path.exists(file_path):
            return ""
        file_name = os.path.basename(file_path)
        backup_path = os.path.join(
            self.backup_dir, f"{file_name}.{os.path.getmtime(file_path):.0f}"
        )
        shutil.copy2(file_path, backup_path)
        return backup_path

    def _restore_backup(self, file_path: str) -> str:
        file_name = os.path.basename(file_path)
        backups = [
            f for f in os.listdir(self.backup_dir) if f.startswith(file_name + ".")
        ]
        if not backups:
            raise FileNotFoundError(f"No backups found for {file_path}")

        latest_backup = sorted(backups, reverse=True)[0]
        backup_path = os.path.join(self.backup_dir, latest_backup)

        shutil.copy2(backup_path, file_path)
        return f"Successfully restored {file_path} from backup"

    def _count_matches(self, content: str, old_str: str) -> int:
        return content.count(old_str)

    def view(self, file_path: str, view_range: Optional[List[int]] = None) -> str:
        try:
            abs_path = self._validate_path(file_path)

            if os.path.isdir(abs_path):
                try:
                    return "\n".join(os.listdir(abs_path))
                except PermissionError:
                    raise PermissionError(
                        "Permission denied. Cannot list directory contents."
                    )

            if not os.path.exists(abs_path):
                raise FileNotFoundError("File not found")

            with open(abs_path, "r", encoding="utf-8") as f:
                content = f.read()

            if view_range:
                start, end = view_range
                lines = content.split("\n")

                if end == -1:
                    end = len(lines)

                selected_lines = lines[start - 1 : end]

                result = []
                for i, line in enumerate(selected_lines, start):
                    result.append(f"{i}: {line}")

                return "\n".join(result)
            else:
                lines = content.split("\n")
                result = []
                for i, line in enumerate(lines, 1):
                    result.append(f"{i}: {line}")

                return "\n".join(result)

        except UnicodeDecodeError:
            raise UnicodeDecodeError(
                "utf-8",
                b"",
                0,
                1,
                "File contains non-text content and cannot be displayed.",
            )
        except ValueError as e:
            raise ValueError(str(e))
        except PermissionError:
            raise PermissionError("Permission denied. Cannot access file.")
        except Exception as e:
            raise type(e)(str(e))

    def str_replace(self, file_path: str, old_str: str, new_str: str) -> str:
        try:
            abs_path = self._validate_path(file_path)

            if not os.path.exists(abs_path):
                raise FileNotFoundError("File not found")

            with open(abs_path, "r", encoding="utf-8") as f:
                content = f.read()

            match_count = self._count_matches(content, old_str)

            if match_count == 0:
                raise ValueError(
                    "No match found for replacement. Please check your text and try again."
                )
            elif match_count > 1:
                raise ValueError(
                    f"Found {match_count} matches for replacement text. Please provide more context to make a unique match."
                )

            # Create backup before modifying
            self._backup_file(abs_path)

            # Perform the replacement
            new_content = content.replace(old_str, new_str)

            with open(abs_path, "w", encoding="utf-8") as f:
                f.write(new_content)

            return "Successfully replaced text at exactly one location."

        except ValueError as e:
            raise ValueError(str(e))
        except PermissionError:
            raise PermissionError("Permission denied. Cannot modify file.")
        except Exception as e:
            raise type(e)(str(e))

    def create(self, file_path: str, file_text: str) -> str:
        try:
            abs_path = self._validate_path(file_path)

            # Check if file already exists
            if os.path.exists(abs_path):
                raise FileExistsError(
                    "File already exists. Use str_replace to modify it."
                )

            # Create parent directories if they don't exist
            os.makedirs(os.path.dirname(abs_path), exist_ok=True)

            # Create the file
            with open(abs_path, "w", encoding="utf-8") as f:
                f.write(file_text)

            return f"Successfully created {file_path}"

        except ValueError as e:
            raise ValueError(str(e))
        except PermissionError:
            raise PermissionError("Permission denied. Cannot create file.")
        except Exception as e:
            raise type(e)(str(e))

    def insert(self, file_path: str, insert_line: int, new_str: str) -> str:
        try:
            abs_path = self._validate_path(file_path)

            if not os.path.exists(abs_path):
                raise FileNotFoundError("File not found")

            # Create backup before modifying
            self._backup_file(abs_path)

            with open(abs_path, "r", encoding="utf-8") as f:
                lines = f.readlines()

            # Handle line endings
            if lines and not lines[-1].endswith("\n"):
                new_str = "\n" + new_str

            # Insert at the beginning if insert_line is 0
            if insert_line == 0:
                lines.insert(0, new_str + "\n")
            # Insert after the specified line
            elif insert_line > 0 and insert_line <= len(lines):
                lines.insert(insert_line, new_str + "\n")
            else:
                raise IndexError(
                    f"Line number {insert_line} is out of range. File has {len(lines)} lines."
                )

            with open(abs_path, "w", encoding="utf-8") as f:
                f.writelines(lines)

            return f"Successfully inserted text after line {insert_line}"

        except ValueError as e:
            raise ValueError(str(e))
        except PermissionError:
            raise PermissionError("Permission denied. Cannot modify file.")
        except Exception as e:
            raise type(e)(str(e))

    def undo_edit(self, file_path: str) -> str:
        try:
            abs_path = self._validate_path(file_path)

            if not os.path.exists(abs_path):
                raise FileNotFoundError("File not found")

            return self._restore_backup(abs_path)

        except ValueError as e:
            raise ValueError(str(e))
        except FileNotFoundError:
            raise FileNotFoundError("No previous edits to undo")
        except PermissionError:
            raise PermissionError("Permission denied. Cannot restore file.")
        except Exception as e:
            raise type(e)(str(e))

In [16]:
# Process Tool Call Requests
import json

text_editor_tool = TextEditorTool()


def run_editor_tool(tool_name, tool_input):
    if tool_name == "str_replace_editor":
        command = tool_input["command"]
        if command == "view":
            return text_editor_tool.view(
                tool_input["path"], tool_input.get("view_range")
            )
        elif command == "str_replace":
            return text_editor_tool.str_replace(
                tool_input["path"], tool_input["old_str"], tool_input["new_str"]
            )
        elif command == "create":
            return text_editor_tool.create(tool_input["path"], tool_input["file_text"])
        elif command == "insert":
            return text_editor_tool.insert(
                tool_input["path"],
                tool_input["insert_line"],
                tool_input["new_str"],
            )
        elif command == "undo_edit":
            return text_editor_tool.undo_edit(tool_input["path"])
        else:
            raise Exception(f"Unknown text editor command: {command}")
    else:
        raise Exception(f"Unknown tool name: {tool_name}")


def run_editor_tools(message):
    tool_requests = [block for block in message.content if block.type == "tool_use"]
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_editor_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False,
            }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True,
            }

        tool_result_blocks.append(tool_result_block)

    return tool_result_blocks

In [17]:
# Make the text edit schema based on the model version being used
def get_text_edit_schema(model):
    return {
        "type": "text_editor_20250728",
        "name": "str_replace_based_edit_tool",
    }

In [28]:
# Run the conversation in a loop until the model doesn't ask for a tool use
def run_editor_tool_usage_conversation(messages):
    while True:
        response = chat(
            client,
            MODEL,
            messages,
            tools=[get_text_edit_schema(MODEL)],
        )

        add_assistant_message(messages, response)
        console.print(
            f"[blue]Got response from chat:[/blue] {response}\n[yellow]-----[/yellow]"
        )
        text_response = text_from_message(response)
        # added this print so we can follow along
        console.print(f"[green]text_from_message:[/green] {text_response}\n")
        console.print(
            f"[red]Stop reason:[/red] {response.stop_reason}\n[yellow]-----[/yellow]"
        )

        if response.stop_reason != "tool_use":
            break

        tool_results = run_editor_tools(response)
        add_user_message(messages, tool_results)

    return messages

In [31]:
import os

console.print(f"[red]os.path.exists(./main.py)?[/red] {os.path.exists('./main.py')}")
# main_filepath = os.path.join(os.getcwd(), "main.py")

# console.print(f"[blue]getcwd:[/blue] {os.getcwd()}")

# console.print(
#     f"[blue]os.path.exists({main_filepath})?[/blue] {os.path.exists(main_filepath)}"
# )

messages = []

add_user_message(
    messages,
    f"""
    Open ./main.py file and summarize its contents.
    """,
)

run_editor_tool_usage_conversation(messages)

os.path.exists(./main.py)? True


Got response from chat: Message(id='msg_011CfLZKknBZADKANoTqj6gT', 
container=None, content=[TextBlock(citations=None, text="I'll open and read the
./main.py file for you.", type='text'), 
ToolUseBlock(id='toolu_01CcGvxyqZ98izC2SZ5UHsC2', 
caller=DirectCaller(type='direct'), input={'command': 'view', 'path': 
'./main.py'}, name='str_replace_based_edit_tool', type='tool_use', 
toolset_name=None)], model='claude-sonnet-4-5-20250929', role='assistant', 
stop_details=None, stop_reason='tool_use', stop_sequence=None, type='message', 
usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, 
ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, 
cache_read_input_tokens=0, inference_geo='not_available', input_tokens=1265, 
output_tokens=94, output_tokens_details=None, server_tool_use=None, 
service_tier='standard'))
-----
text_from_message: I'll open and read the ./main.py file for you.

Stop reason: tool_use
-----
Got response from chat: Message(id='msg_011CfLZKudVUY1TYX7

[{'role': 'user',
  'content': '\n    Open ./main.py file and summarize its contents.\n    '},
 {'role': 'assistant',
  'content': [{'type': 'text',
    'text': "I'll open and read the ./main.py file for you."},
   {'type': 'tool_use',
    'id': 'toolu_01CcGvxyqZ98izC2SZ5UHsC2',
    'name': 'str_replace_based_edit_tool',
    'input': {'command': 'view', 'path': './main.py'}}]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01CcGvxyqZ98izC2SZ5UHsC2',
    'content': 'Error: Unknown tool name: str_replace_based_edit_tool',
    'is_error': True}]},
 {'role': 'assistant',
  'content': [{'type': 'tool_use',
    'id': 'toolu_01K61KSi6iYA7ZrCeq9iz2MJ',
    'name': 'str_replace_based_edit_tool',
    'input': {'command': 'view', 'path': './main.py'}}]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01K61KSi6iYA7ZrCeq9iz2MJ',
    'content': 'Error: Unknown tool name: str_replace_based_edit_tool',
    'is_error': True}]},
 {'r